# OpenAI Batch API — Prueba ABSA

Notebook standalone para probar la misma configuración de prompt/modelo que usa el `ABSAService`.
Flujo:
1. Cargar CSV con reviews
2. Definir tópicos fijos (y opcionales adicionales)
3. Construir JSONL y enviar batch a OpenAI
4. Esperar y descargar resultados
5. Parsear y visualizar

## 1. Configuración

In [ ]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "XXXX")
OPENAI_MODEL="gpt-5.4-mini"  # o pégala aquí
#OPENAI_MODEL   = "gpt-4o-mini"                          # mismo default que ABSAService

# Columna del CSV que contiene el texto de la review
COLUMNA_TEXTO  = "texto_limpio"  # cámbiala si tu CSV usa otro nombre
# Columna id (opcional — si no existe se usará el índice)
COLUMNA_ID     = "id"  # ej. "review_id"

# Límite de reviews a enviar (None = todas)
LIMITE_REVIEWS = None

## 2. Tópicos

Copia aquí los tópicos fijos tal como están en la BD.
Cada entrada: `{"id": int, "slug": str, "nombre": str}`.

In [ ]:
TOPICOS_FIJOS = [
    {"id": 1, "slug": "limpieza",          "nombre": "Limpieza"},
    {"id": 2, "slug": "servicio",           "nombre": "Servicio / Atención"},
    {"id": 3, "slug": "habitacion",         "nombre": "Habitación"},
    {"id": 4, "slug": "desayuno-comida",    "nombre": "Desayuno / Comida"},
    {"id": 5, "slug": "precio-valor",       "nombre": "Precio / Relación calidad-precio"},
    {"id": 6, "slug": "ubicacion",          "nombre": "Ubicación"},
    {"id": 7, "slug": "instalaciones",      "nombre": "Instalaciones / Comodidades"},
]

# Tópicos adicionales ya descubiertos (puede quedar vacío)
TOPICOS_ADICIONALES = [
    # {"slug": "piscina", "nombre": "Piscina"},
]

## 3. Cargar CSV

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# ── Carga ─────────────────────────────────────────────────────────────────────
CSV_PATH = Path("../sources/data/reviews_review_topicos_202604201935.csv")  # <-- ajusta la ruta

df = pd.read_csv(CSV_PATH)
print(f"Columnas: {list(df.columns)}")
print(f"Filas totales: {len(df)}")
display(df.head(3))

In [ ]:
# ── Preparar lista de reviews ─────────────────────────────────────────────────
if COLUMNA_TEXTO not in df.columns:
    raise ValueError(f"Columna '{COLUMNA_TEXTO}' no encontrada. Disponibles: {list(df.columns)}")

df_work = df.dropna(subset=[COLUMNA_TEXTO]).copy()
if LIMITE_REVIEWS:
    df_work = df_work.head(LIMITE_REVIEWS)

if COLUMNA_ID and COLUMNA_ID in df_work.columns:
    df_work["_review_id"] = df_work[COLUMNA_ID].astype(str)
else:
    df_work["_review_id"] = df_work.index.astype(str)

reviews = df_work[["_review_id", COLUMNA_TEXTO]].rename(columns={COLUMNA_TEXTO: "texto"}).to_dict("records")
print(f"Reviews a procesar: {len(reviews)}")
print("Primer texto:", reviews[0]["texto"][:120])

## 4. Constructor de prompt (mismo que `prompt_builder.py`)

In [ ]:
_PLANTILLA = """\
Eres un analista de opiniones de hoteles. Lee la review en español y extrae tópicos.

Tópicos predefinidos (usa nombre y slug EXACTOS):
{topicos}
{seccion_adicionales}

Objetivo:
- Priorizar tópicos predefinidos.
- Permitir tópicos nuevos cuando aporten información realmente distinta.

Reglas de decisión:
1) Prioriza mapear a predefinidos.
2) Reutiliza adicionales existentes cuando sean equivalentes.
3) Crea tópico nuevo solo si representa un aspecto claramente distinto y útil.
4) Si hay duda entre nuevo y predefinido, elige predefinido.

Guía de mapeo recomendada:
- instalaciones_servicios: mantenimiento, agua caliente, baño, ascensor, estado de instalaciones.
- atencion_cliente: trato, recepción, servicio.
- desayuno_gastronomia: desayuno/comida.
- comodidad: descanso, cama, almohadas.
- limpieza: higiene/suciedad.
- calidad_precio: percepción de valor/precio.

Extracción:
- Varios tópicos por review si aplica.
- Fragmento literal (20–300 caracteres).
- sentimiento_topico: positivo | negativo | neutro por fragmento.
- score_topico: 0.000 a 1.000.
- Evita duplicados semánticos.

Regla para tópicos nuevos:
- Solo si no encaja razonablemente en predefinidos ni adicionales.
- Requiere score_topico >= 0.80.
- Si es nuevo: slug="" y topico breve, específico y no redundante.

Responde SOLO JSON válido.

TEXTO DE LA REVIEW:
{texto}

Formato obligatorio:
{{"items":[{{"topico":"...","slug":"...","fragmento":"...","sentimiento_topico":"positivo|negativo|neutro","score_topico":0.000}}]}}"""

_SECCION_ADICIONALES = """\
Tópicos adicionales ya descubiertos (si la review menciona algo equivalente, \
REUTILIZA uno de estos con su slug EXACTO en lugar de crear uno nuevo):
{adicionales}
"""

_SECCION_ADICIONALES = """\
Tópicos adicionales ya descubiertos (si la review menciona algo equivalente, \
REUTILIZA uno de estos con su slug EXACTO en lugar de crear uno nuevo):
{adicionales}
"""


def construir_prompt(texto, topicos_fijos, topicos_adicionales=None):
    lineas_fijos = [f"- {t['nombre']} (slug: {t['slug']})" for t in topicos_fijos]
    if topicos_adicionales:
        lineas_ad = [f"- {t['nombre']} (slug: {t['slug']})" for t in topicos_adicionales]
        seccion = _SECCION_ADICIONALES.format(adicionales="\n".join(lineas_ad))
    else:
        seccion = ""
    return _PLANTILLA.format(
        topicos="\n".join(lineas_fijos),
        seccion_adicionales=seccion,
        texto=texto.strip(),
    )


# Vista previa del prompt
print(construir_prompt(reviews[0]["texto"], TOPICOS_FIJOS, TOPICOS_ADICIONALES or None))

## 5. Llamar API directamente

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
print("Cliente OpenAI listo.")

In [ ]:
from tqdm.auto import tqdm

resultados_api = []
tokens_entrada_total = 0
tokens_salida_total  = 0
errores = 0

for r in tqdm(reviews, desc="Analizando reviews"):
    try:
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{"role": "user", "content": construir_prompt(r["texto"], TOPICOS_FIJOS, TOPICOS_ADICIONALES or None)}],
            temperature=0,
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content
        uso = response.usage
        t_in  = uso.prompt_tokens     if uso else 0
        t_out = uso.completion_tokens if uso else 0
    except Exception as e:
        print(f"Error review_id={r['_review_id']}: {e}")
        content, t_in, t_out = None, 0, 0
        errores += 1

    tokens_entrada_total += t_in
    tokens_salida_total  += t_out
    resultados_api.append({
        "review_id": r["_review_id"],
        "content":   content,
        "tokens_in":  t_in,
        "tokens_out": t_out,
    })
    print(response)

print(f"\nReviews procesadas: {len(resultados_api) - errores}/{len(reviews)}")
print(f"Errores:            {errores}")
print(f"Tokens entrada:     {tokens_entrada_total:,}")
print(f"Tokens salida:      {tokens_salida_total:,}")

## 6. Parsear y visualizar resultados

In [ ]:
import re

_SENTIMIENTOS = {"positivo", "negativo", "neutro"}


def _extraer_json(raw):
    sin_md = re.sub(r"```(?:json)?\s*", "", raw).replace("```", "").strip()
    inicio = sin_md.find("{")
    fin    = sin_md.rfind("}")
    if inicio == -1 or fin == -1 or fin < inicio:
        return None
    return sin_md[inicio:fin + 1]


def parsear_respuesta(raw, slugs_esperados):
    texto_json = _extraer_json(raw)
    if not texto_json:
        return None
    try:
        data = json.loads(texto_json)
    except json.JSONDecodeError:
        return None
    items = data.get("items", []) if isinstance(data, dict) else []
    fijos, dinamicos = [], []
    slugs_vistos = set()
    for item in items:
        if not isinstance(item, dict):
            continue
        slug        = str(item.get("slug") or "").strip()
        nombre      = str(item.get("topico") or "").strip()
        fragmento   = str(item.get("fragmento") or "").strip()[:500]
        sentimiento = str(item.get("sentimiento_topico") or "neutro").lower()
        if sentimiento not in _SENTIMIENTOS:
            sentimiento = "neutro"
        raw_score = item.get("score_topico")
        score = float(raw_score) if raw_score is not None else 0.5
        score = max(0.0, min(1.0, score))
        if slug in slugs_esperados:
            slugs_vistos.add(slug)
            fijos.append({"slug": slug, "mencionado": True, "sentimiento": sentimiento,
                          "score": score, "fragmento": fragmento})
        elif nombre:
            dinamicos.append({"nombre": nombre[:120], "slug": slug, "sentimiento": sentimiento,
                               "score": score, "fragmento": fragmento})
    for slug in slugs_esperados:
        if slug not in slugs_vistos:
            fijos.append({"slug": slug, "mencionado": False, "sentimiento": None,
                          "score": None, "fragmento": None})
    return {"topicos_fijos": fijos, "topicos_dinamicos": dinamicos}

In [ ]:
slugs_esperados = {t["slug"] for t in TOPICOS_FIJOS}

filas = []
lineas_ok = 0
lineas_fallidas = 0

for r in resultados_api:
    review_id = r["review_id"]
    content   = r["content"]
    t_in      = r["tokens_in"]
    t_out     = r["tokens_out"]

    if content is None:
        lineas_fallidas += 1
        continue

    parsed = parsear_respuesta(content, slugs_esperados)
    if parsed is None:
        lineas_fallidas += 1
        continue

    for tf in parsed["topicos_fijos"]:
        if tf["mencionado"]:
            filas.append({
                "review_id":   review_id,
                "tipo":        "fijo",
                "topico":      tf["slug"],
                "sentimiento": tf["sentimiento"],
                "score":       tf["score"],
                "fragmento":   tf["fragmento"],
                "tokens_in":   t_in,
                "tokens_out":  t_out,
            })
    for td in parsed["topicos_dinamicos"]:
        filas.append({
            "review_id":   review_id,
            "tipo":        "dinamico",
            "topico":      td["nombre"],
            "sentimiento": td["sentimiento"],
            "score":       td["score"],
            "fragmento":   td["fragmento"],
            "tokens_in":   t_in,
            "tokens_out":  t_out,
        })
    lineas_ok += 1

df_resultados = pd.DataFrame(filas)

print(f"Reviews OK:           {lineas_ok}")
print(f"Reviews fallidas:     {lineas_fallidas}")
print(f"Asignaciones totales: {len(filas)}")
display(df_resultados.head(10))

## 8. Visualización rápida

In [ ]:
import matplotlib.pyplot as plt

# ── Distribución de tópicos mencionados ──────────────────────────────────────
dist = df_resultados["topico"].value_counts()
fig, ax = plt.subplots(figsize=(10, 4))
dist.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Menciones por tópico")
ax.set_ylabel("Nº de menciones")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ── Sentimiento por tópico (solo fijos) ──────────────────────────────────────
df_fijos = df_resultados[df_resultados["tipo"] == "fijo"]
tabla = (
    df_fijos.groupby(["topico", "sentimiento"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["positivo", "negativo", "neutro"], fill_value=0)
)

tabla.plot(kind="bar", figsize=(12, 5), color=["#4caf50", "#f44336", "#9e9e9e"])
plt.title("Distribución de sentimiento por tópico fijo")
plt.ylabel("Nº de menciones")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

display(tabla)

In [ ]:
# ── Tópicos dinámicos descubiertos ───────────────────────────────────────────
df_din = df_resultados[df_resultados["tipo"] == "dinamico"]
print(f"Tópicos dinámicos únicos: {df_din['topico'].nunique()}")
display(
    df_din.groupby("topico")
    .agg(menciones=("review_id", "count"), sentimientos=("sentimiento", lambda x: x.value_counts().to_dict()))
    .sort_values("menciones", ascending=False)
    .head(20)
)

In [ ]:
from datetime import datetime

OUT_PATH = Path(f"resultados_absa_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
df_resultados.to_csv(OUT_PATH, index=False)
print(f"Guardado en: {OUT_PATH}")